In [ ]:
import numpy as np
import random
from multiprocessing import Pool
import time

# Problem setup
NUM_NODES = 10
NUM_SOURCES = 3
HOURS = 24

POP_SIZE = 20
GENERATIONS = 30
MUTATION_RATE = 0.1

# Create random demand (10 nodes x 24 hours)
demand = np.random.randint(20, 50, size=(NUM_NODES, HOURS))

# Source capacities (3 sources)
source_capacity = [500, 400, 300]  # Total energy available per hour per source

# Chromosome = allocation matrix: nodes x hours x source
def create_individual():
    return np.random.dirichlet(np.ones(NUM_SOURCES), size=(NUM_NODES, HOURS))

# Fitness: reward meeting demand, penalize overuse of capacity and mismatch
def fitness(individual):
    fitness_score = 0
    source_usage = np.zeros((NUM_SOURCES, HOURS))

    for n in range(NUM_NODES):
        for h in range(HOURS):
            node_demand = demand[n][h]
            allocations = individual[n][h] * node_demand
            source_usage[:, h] += allocations
            fitness_score += -abs(np.sum(allocations) - node_demand)  # Penalize mismatch

    for s in range(NUM_SOURCES):
        for h in range(HOURS):
            if source_usage[s][h] > source_capacity[s]:
                fitness_score -= (source_usage[s][h] - source_capacity[s]) * 5  # Heavy penalty

    return fitness_score

# Parallel fitness evaluation
def evaluate_population(pop):
    with Pool() as pool:
        return pool.map(fitness, pop)

# Selection
def select(pop, fitnesses):
    idx = np.argsort(fitnesses)[-POP_SIZE//2:]
    return [pop[i] for i in idx]

# Crossover
def crossover(parent1, parent2):
    child = np.copy(parent1)
    mask = np.random.rand(NUM_NODES, HOURS, NUM_SOURCES) > 0.5
    child[mask] = parent2[mask]
    return child

# Mutation
def mutate(individual):
    for _ in range(int(NUM_NODES * HOURS * MUTATION_RATE)):
        n = random.randint(0, NUM_NODES-1)
        h = random.randint(0, HOURS-1)
        individual[n][h] = np.random.dirichlet(np.ones(NUM_SOURCES))
    return individual

# Main GA loop
def run_ga(parallel=True):
    pop = [create_individual() for _ in range(POP_SIZE)]
    for gen in range(GENERATIONS):
        start = time.time()
        fitnesses = evaluate_population(pop) if parallel else [fitness(ind) for ind in pop]
        elapsed = time.time() - start
        best_fit = max(fitnesses)
        print(f"Gen {gen+1}: Best Fitness = {best_fit:.2f} | Time = {elapsed:.3f}s")

        selected = select(pop, fitnesses)
        new_pop = selected.copy()
        while len(new_pop) < POP_SIZE:
            p1, p2 = random.sample(selected, 2)
            child = crossover(p1, p2)
            new_pop.append(mutate(child))
        pop = new_pop
    return pop

# Run both versions
print("=== Serial GA ===")
start_serial = time.time()
run_ga(parallel=False)
serial_time = time.time() - start_serial

print("\n=== Parallel GA ===")
start_parallel = time.time()
run_ga(parallel=True)
parallel_time = time.time() - start_parallel

print(f"\nSerial Time: {serial_time:.2f}s | Parallel Time: {parallel_time:.2f}s")


=== Serial GA ===
Gen 1: Best Fitness = -0.00 | Time = 0.064s
Gen 2: Best Fitness = -0.00 | Time = 0.064s
Gen 3: Best Fitness = -0.00 | Time = 0.068s
Gen 4: Best Fitness = -0.00 | Time = 0.066s
Gen 5: Best Fitness = -0.00 | Time = 0.065s
Gen 6: Best Fitness = -0.00 | Time = 0.068s
Gen 7: Best Fitness = -0.00 | Time = 0.069s
Gen 8: Best Fitness = -0.00 | Time = 0.071s
Gen 9: Best Fitness = -0.00 | Time = 0.072s
Gen 10: Best Fitness = -0.00 | Time = 0.083s
Gen 11: Best Fitness = -0.00 | Time = 0.068s
Gen 12: Best Fitness = -0.00 | Time = 0.114s
Gen 13: Best Fitness = -0.00 | Time = 0.069s
Gen 14: Best Fitness = -0.00 | Time = 0.085s
Gen 15: Best Fitness = -0.00 | Time = 0.043s
Gen 16: Best Fitness = -0.00 | Time = 0.037s
Gen 17: Best Fitness = -0.00 | Time = 0.036s
Gen 18: Best Fitness = -0.00 | Time = 0.064s
Gen 19: Best Fitness = -0.00 | Time = 0.088s
Gen 20: Best Fitness = -0.00 | Time = 0.073s
Gen 21: Best Fitness = -0.00 | Time = 0.077s
Gen 22: Best Fitness = -0.00 | Time = 0.052s
G